# ✈️ Flight Price Prediction

An end-to-end supervised-learning project: predict the **fare of a domestic Indian flight** from its
route, carrier, schedule, duration and booking window.

- **Dataset** — 300,153 domestic flights (6 Indian carriers, 6 metros)
- **Model** — `RandomForestRegressor` wrapped in a `ColumnTransformer` pipeline
- **Output** — fare in INR, with an MAE of roughly **₹1,300**
- **App** — a dark-themed Streamlit front end in `app/app.py`

Everything below is reproducible: `scripts/train_model.py` in the repository root is this notebook
in script form and writes the two artifacts the app consumes.


In [ ]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def find_root(start: Path) -> Path:
    """Walk up until the dataset folder shows up, so the notebook runs from
    either the repository root or the notebooks/ directory."""
    for folder in [start, *start.parents]:
        if (folder / "data" / "Clean_Dataset.csv").exists():
            return folder
    raise FileNotFoundError("data/Clean_Dataset.csv not found above " + str(start))


BASE_DIR = find_root(Path.cwd())
DATA_PATH = BASE_DIR / "data" / "Clean_Dataset.csv"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
print("project root:", BASE_DIR)

## 1. Load the data

`data/Clean_Dataset.csv` holds one row per flight. The unnamed first column is a leftover index and
the `flight` column is a carrier-specific flight number — see section 2 for why it is dropped.


In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")

print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

## 2. The feature contract

Nine columns feed the model: seven categorical and two numerical.

The raw dataset also carries `flight` (1,561 distinct flight numbers). It is deliberately
**excluded**: it is a high-cardinality identifier, one-hot encoding it would add 1,561 columns, and
it would not generalise to any flight number the model has not already seen.


In [ ]:
TARGET = "price"

CATEGORICAL_FEATURES = [
    "airline",
    "source_city",
    "departure_time",
    "stops",
    "arrival_time",
    "destination_city",
    "class",
]

NUMERICAL_FEATURES = ["duration", "days_left"]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

print("features :", FEATURES)
print("dropped  : ['flight'] (identifier)")
print()
for col in CATEGORICAL_FEATURES:
    print(f"{col:<18} {sorted(df[col].astype(str).unique())}")
for col in NUMERICAL_FEATURES:
    print(f"{col:<18} {df[col].min()} .. {df[col].max()}")

## 3. What the data looks like

Fares are heavily right-skewed: the median is ₹7,425 but the mean is ₹20,890, because business
class pulls the average far above the typical economy ticket.


In [ ]:
print(f"price   min ₹{df.price.min():,.0f}   median ₹{df.price.median():,.0f}   mean ₹{df.price.mean():,.0f}   max ₹{df.price.max():,.0f}")
print()
print(df["class"].value_counts().to_frame("count").assign(share=lambda d: (d["count"]/len(df)*100).round(1)))
print()
print(df["stops"].value_counts().to_frame("count").assign(share=lambda d: (d["count"]/len(df)*100).round(1)))

## 4. Split

A stratified-free 80/20 split. The test set is only ever touched once, at evaluation time.


In [ ]:
X = df[FEATURES]
y = df[TARGET].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"train {len(X_train):,} | test {len(X_test):,}")

## 5. The pipeline

Preprocessing lives **inside** the pipeline so the saved artifact is self-contained: the app only
has to hand it a DataFrame with the right column names.

`OneHotEncoder(handle_unknown="ignore")` means an unseen category maps to all-zeros instead of
raising — the app should never crash on an unexpected value.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("num", "passthrough", NUMERICAL_FEATURES),
    ],
    remainder="drop",
)

regressor = RandomForestRegressor(
    n_estimators=200,
    max_depth=30,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)

pipeline = Pipeline([("preprocessor", preprocessor), ("regressor", regressor)])

## 6. Train


In [ ]:
start = time.time()
pipeline.fit(X_train, y_train)
print(f"trained in {time.time() - start:.1f}s")

## 7. Evaluate

The MAE confidence interval comes from a percentile bootstrap over 200 resamples of the test errors.


In [ ]:
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

rng = np.random.default_rng(42)
errors = np.abs(y_test.values - y_pred)
means = np.array([rng.choice(errors, size=len(errors), replace=True).mean() for _ in range(200)])
lo, hi = np.percentile(means, [2.5, 97.5])

print(f"MAE   ₹{mae:,.2f}   CI95 [₹{lo:,.2f}, ₹{hi:,.2f}]")
print(f"RMSE  ₹{rmse:,.2f}")
print(f"R²    {r2:.6f}")

## 8. What actually drives the price

Collapsing the one-hot columns back onto their parent feature gives a surprisingly blunt picture:
**the travel class carries 88% of the signal**, and the route and airline barely register.


In [ ]:
encoded = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["regressor"].feature_importances_

totals = {f: 0.0 for f in FEATURES}
for name, value in zip(encoded, importances):
    bare = name.split("__", 1)[1]
    parent = next((f for f in FEATURES if bare == f or bare.startswith(f + "_")), None)
    if parent:
        totals[parent] += float(value)

for feature, share in sorted(totals.items(), key=lambda kv: -kv[1]):
    print(f"{feature:<20} {share*100:6.2f}%")

## 9. The booking curve

Fares fall steeply as the departure date approaches... in reverse: the further out you book, the
cheaper the seat. Economy fares drop about 67% between one day before departure and seven weeks
out, business fares about 22%.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, klass in zip(axes, ["Economy", "Business"]):
    mean = df[df["class"] == klass].groupby("days_left")["price"].mean()
    ax.plot(mean.index, mean.values, linewidth=2.4)
    ax.set_title(klass)
    ax.set_xlabel("Days before departure")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Average fare (INR)")
plt.tight_layout()
plt.show()

## 10. Persist the artifacts

Two files come out of this, and they are exactly what the Streamlit app loads:

- `artifacts/flight_price_pipeline.joblib` — the fitted pipeline (preprocessing + model)
- `artifacts/flight_price_model.meta.json` — the feature contract, the category lists and the metrics

> The pipeline is around **1 GB**, which is above GitHub's 100 MB per-file limit, so it is not
> committed. Run this notebook (or `python scripts/train_model.py`) to regenerate it locally.


In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline, ARTIFACTS_DIR / "flight_price_pipeline.joblib")

metadata = {
    "model_name": "Random Forest",
    "target": TARGET,
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numerical_features": NUMERICAL_FEATURES,
    "metrics": {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAE_CI_95_lower": lo,
        "MAE_CI_95_upper": hi,
    },
    "categorical_options": {
        col: sorted(df[col].astype(str).unique().tolist()) for col in CATEGORICAL_FEATURES
    },
}

with open(ARTIFACTS_DIR / "flight_price_model.meta.json", "w", encoding="utf-8") as fh:
    json.dump(metadata, fh, indent=4, ensure_ascii=False)
    fh.write("\n")

print("saved artifacts")

## 11. Predict

The app builds a single-row DataFrame using the feature order from the metadata and calls
`pipeline.predict`. Here is the same thing, from the notebook.


In [ ]:
def predict_fare(airline, source_city, destination_city, departure_time, arrival_time,
                  stops, travel_class, duration, days_left):
    values = {
        "airline": airline,
        "source_city": source_city,
        "departure_time": departure_time,
        "stops": stops,
        "arrival_time": arrival_time,
        "destination_city": destination_city,
        "class": travel_class,
        "duration": duration,
        "days_left": days_left,
    }
    frame = pd.DataFrame([[values[col] for col in FEATURES]], columns=FEATURES)
    return float(pipeline.predict(frame)[0])


for days in (1, 7, 15, 30, 45):
    price = predict_fare("Indigo", "Delhi", "Mumbai", "Afternoon", "Evening",
                         "zero", "Economy", 2.17, days)
    print(f"Delhi -> Mumbai, {days:>2} days ahead:  ₹{price:>9,.2f}")